# Ejercicio 8 - Incorporacion de datos de 2025

## 8.1 Modificacion del sistema de descarga

En `scripts/download_data.py` se amplio `ANIOS_POR_DEFECTO` de `(2024, 2026)` a `(2024, 2025, 2026)` y se actualizo la descripcion de uso. Se descarga taxis amarillos (`yellow`) y verdes (`green`) y conserva los parametros `--anio`, `--taxi` y `--solo-verificar`


### Para ejecutar 

```bash
# Incorporar exclusivamente 2025, sin recorrer los otros anios.
docker exec lab8-lab python scripts/download_data.py --anio 2025

# Descarga predeterminada: 2024, 2025 y 2026.
docker exec lab8-lab python scripts/download_data.py

# Seleccion explicita equivalente.
docker exec lab8-lab python scripts/download_data.py --anio 2024 2025 2026
```

Los archivos de 2025 se guardan en `data/raw/yellow/2025/` y `data/raw/green/2025/`, con nombres como `yellow_tripdata_2025-01.parquet` y `green_tripdata_2025-01.parquet`.

```bash
docker exec lab8-lab python scripts/download_data.py --anio 2025 --solo-verificar
```

La verificacion compara los archivos locales con el servidor y muestra un inventario por tipo, con archivos, filas y tamaños


## 8.2 Verificacion de archivos previamente descargados

Se reviso `scripts/download_data.py` y se repitio la descarga normal de 2025 en Docker, sin `--solo-verificar`. Esto comprueba que el flujo que podria descargar archivos los omite cuando ya existen

```bash
docker exec lab8-lab python scripts/download_data.py --anio 2025
```

Antes y despues se detectaron 24 archivos Parquet: 12 yellow y 12 green, correspondientes a enero-diciembre de 2025. El comando termino con codigo de salida 0 y mostro `ya existe, se omite` para cada uno de los 24 meses/tipos.

Extracto de la salida observada:

```text
=== YELLOW 2025 ===
  2025-01  ya existe, se omite
  2025-12  ya existe, se omite

=== GREEN 2025 ===
  2025-01  ya existe, se omite
  2025-12  ya existe, se omite

RESUMEN
  descargados   : 0
  actualizados  : 0
  ya existian   : 24
  no publicados : 0
  fallidos      : 0
```

El extracto abrevia las lineas intermedias; todos los meses 01-12 se omitieron en ambos tipos. La tabla `taxi_zone_lookup.csv` tambien se omitio y no forma parte del contador de 24 Parquet.

| Comprobacion | Resultado |
|--------------|-----------|
| Archivos existentes detectados | 24 |
| Archivos omitidos | 24 |
| Archivos descargados nuevamente | 0 |
| Archivos actualizados | 0 |
| Pendientes | 0; ningun archivo mensual falto ni requirio descarga |
| Fallidos | 0 |
| Meses no publicados | 0 |



## 8.3 Verificacion de consultas sobre el conjunto ampliado


La inspeccion inicial de `carga` y `viajes` encontro 40 archivos y 71,870,407 registros, correspondientes solo a 2024 y 2026. 
```bash
docker exec lab8-lab python scripts/materializar.py --base data/processed/taxis_ampliado_8_3.duckdb
```

Una vez validada la carga, se detiene temporalmente Metabase para liberar la base. Se conserva la base previa como `data/processed/taxis_antes_2025_8_3.duckdb` y se coloca la nueva en `data/processed/taxis.duckdb`, antes de reiniciar Metabase. No se eliminan los Parquet ni se modifican sus datos.

La celda siguiente abre la base actualizada en solo lectura mediante `run_sql.conectar`, ejecuta cada archivo SQL original completo y registra filas, columnas, anios y series. Los anios se obtienen de `anio`, de las etiquetas `serie` o del prefijo de `mes`, segun la salida. En I2 incluyen tanto el año evaluado como el año base. Si hay error para.


```bash
docker exec lab8-lab jupyter nbconvert --to notebook --execute --inplace notebooks/08_incorporacion2025.ipynb --ExecutePreprocessor.timeout=600
```

In [1]:
from pathlib import Path
import re
import sys
import time
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
from run_sql import conectar

con = conectar(base=str(RAIZ / "data/processed/taxis.duckdb"), solo_lectura=True)
try:
    carga = con.sql("SELECT * FROM carga").df()
    cobertura = con.sql("SELECT anio, count(*) AS registros, count(DISTINCT mes_archivo) AS meses FROM viajes GROUP BY anio ORDER BY anio").df()
    meses_comunes = con.sql("SELECT substr(mes_archivo, 6, 2) AS mes FROM viajes GROUP BY mes HAVING count(DISTINCT anio) = (SELECT count(DISTINCT anio) FROM viajes) ORDER BY mes").df()["mes"].tolist()
    assert cobertura["anio"].tolist() == [2024, 2025, 2026], "La base no contiene los tres anios esperados"
    print("Carga materializada:")
    display(carga)
    print("Cobertura de la base:")
    display(cobertura)
    print("Meses comunes:", meses_comunes)
    resultados = []
    for ruta in sorted((RAIZ / "sql/07_indicadores").glob("*.sql")):
        inicio = time.perf_counter()
        df = con.sql(ruta.read_text(encoding="utf-8")).df()
        anios = set()
        for columna in ("anio", "serie", "mes"):
            if columna in df.columns:
                for valor in df[columna].dropna().astype(str):
                    anios.update(int(a) for a in re.findall(r"\b20\d{2}\b", valor))
        assert len(df) > 0, f"Consulta sin filas: {ruta.name}"
        assert anios == {2024, 2025, 2026}, f"Anios inesperados en {ruta.name}: {anios}"
        series = sorted(df["serie"].unique().tolist()) if "serie" in df.columns else []
        resultados.append({"indicador": "I" + str(int(ruta.name[:2])), "archivo": ruta.name,
                           "estado": "Correcto", "filas": len(df), "anios": ", ".join(map(str, sorted(anios))),
                           "segundos": round(time.perf_counter() - inicio, 2), "series": "; ".join(series)})
        print(resultados[-1])
    assert len(resultados) == 11, "Se esperaban once consultas"
    resumen_8_3 = pd.DataFrame(resultados)
    display(resumen_8_3)
finally:
    con.close()

Carga materializada:


,fecha_carga,periodo,archivos,filas,segundos,anios,ultimo_mes
0,2026-10-09 02:28:15.779930,*/*,64,121184384,45.958828,"[2024, 2025, 2026]",2026-08


Cobertura de la base:


,anio,registros,meses
0,2024,41829938,12
1,2025,49313977,12
2,2026,30040469,8


Meses comunes: ['01', '02', '03', '04', '05', '06', '07', '08']


...

{'indicador': 'I8', 'archivo': '08_velocidad_manhattan.sql', 'estado': 'Correcto', 'filas': 72, 'anios': '2024, 2025, 2026', 'segundos': 32.0, 'series': ''}


{'indicador': 'I9', 'archivo': '09_zonas_de_origen.sql', 'estado': 'Correcto', 'filas': 30, 'anios': '2024, 2025, 2026', 'segundos': 1.51, 'series': ''}


{'indicador': 'I10', 'archivo': '10_aeropuertos.sql', 'estado': 'Correcto', 'filas': 18, 'anios': '2024, 2025, 2026', 'segundos': 8.09, 'series': '2024 % ingresos; 2024 % viajes; 2025 % ingresos; 2025 % viajes; 2026 % ingresos; 2026 % viajes'}


{'indicador': 'I11', 'archivo': '11_calidad_de_datos.sql', 'estado': 'Correcto', 'filas': 64, 'anios': '2024, 2025, 2026', 'segundos': 1.42, 'series': ''}


,indicador,archivo,estado,filas,anios,segundos,series
0,I1,01_viajes_por_mes.sql,Correcto,32,"2024, 2025, 2026",0.39,
1,I2,02_variacion_interanual.sql,Correcto,40,"2024, 2025, 2026",0.52,green 2025 vs 2024; green 2026 vs 2025; yellow...
2,I3,03_costo_por_viaje.sql,Correcto,32,"2024, 2025, 2026",27.82,
3,I4,04_composicion_del_cobro.sql,Correcto,18,"2024, 2025, 2026",5.58,
4,I5,05_metodos_de_pago.sql,Correcto,128,"2024, 2025, 2026",1.23,
5,I6,06_propina_con_tarjeta.sql,Correcto,32,"2024, 2025, 2026",1.95,
6,I7,07_demanda_por_hora.sql,Correcto,144,"2024, 2025, 2026",5.89,2024 fin de semana; 2024 laborable; 2025 fin d...
7,I8,08_velocidad_manhattan.sql,Correcto,72,"2024, 2025, 2026",32.00,
8,I9,09_zonas_de_origen.sql,Correcto,30,"2024, 2025, 2026",1.51,
9,I10,10_aeropuertos.sql,Correcto,18,"2024, 2025, 2026",8.09,2024 % ingresos; 2024 % viajes; 2025 % ingreso...




| Indicador | Ejecucion | Filas | Anios presentes | Comportamiento con 2025 |
|-----------|-----------|------:|-----------------|-------------------------|
| I1 | Correcta | 32 | 2024, 2025, 2026 | Agrega los 12 meses de 2025. |
| I2 | Correcta | 40 | 2025 vs 2024 y 2026 vs 2025 | `lag()` usa 2024 como base de 2025 (12 meses) y 2025 como base de 2026 (8 meses), para yellow y green. 2024 no tiene comparacion propia por carecer de un anio anterior cargado. |
| I3 | Correcta | 32 | 2024, 2025, 2026 | Agrega resultados mensuales de 2025; conserva yellow y total mayor que cero. |
| I4 | Correcta | 18 | 2024, 2025, 2026 | Seis componentes por anio; conserva enero-agosto como meses comunes. |
| I5 | Correcta | 128 | 2024, 2025, 2026 | Cuatro categorias de pago en los 32 meses disponibles. El anio esta en el prefijo de `mes`. |
| I6 | Correcta | 32 | 2024, 2025, 2026 | Agrega los meses de 2025; conserva el filtro de pago con tarjeta. |
| I7 | Correcta | 144 | 2024, 2025, 2026 | 24 horas por dos tipos de dia y tres anios; utiliza meses comunes y fechas del anio correspondiente. |
| I8 | Correcta | 72 | 2024, 2025, 2026 | 24 horas por tres anios; conserva meses comunes y filtros de Manhattan, yellow, dias laborables, duracion y velocidad. |
| I9 | Correcta | 30 | 2024, 2025, 2026 | Diez zonas por tres anios; `max(anio)` sigue seleccionando 2026 para definir el top. |
| I10 | Correcta | 18 | 2024, 2025, 2026 | Tres aeropuertos por dos medidas y tres anios; conserva meses comunes y asignacion prioritaria al aeropuerto de origen. |
| I11 | Correcta | 64 | 2024, 2025, 2026 | Dos tipos de taxi en los 32 meses disponibles; el anio esta en el prefijo de `mes`. |



## 8.4 Actualizacion de indicadores y visualizaciones para tres años


```bash
docker exec --env-file .env lab8-lab python scripts/metabase_indicadores.py
docker exec --env-file .env lab8-lab python scripts/metabase_indicadores.py --solo-tablero
```

El primer comando actualiza las preguntas por nombre y ejecuta cada consulta. El segundo actualiza la distribucion del tablero reutilizando las preguntas y las tarjetas existentes. El tablero sigue disponible en <http://localhost:3000/dashboard/2>.


Para ejecutar el notebook desde Docker con acceso a Metabase:

```bash
docker exec --env-file .env lab8-lab jupyter nbconvert --to notebook --execute --inplace notebooks/08_incorporacion2025.ipynb --ExecutePreprocessor.timeout=600
```


In [1]:
from pathlib import Path
import os
import re
import sys
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
from metabase_indicadores import Metabase, INDICADORES, NOMBRE_COLECCION, NOMBRE_TABLERO, elementos_coleccion

mb = Metabase(os.environ.get("MB_URL", "http://metabase:3000"))
mb.iniciar_sesion(os.environ["MB_USER"], os.environ["MB_PASSWORD"])
colecciones = [c for c in mb.pedir("GET", "/api/collection") if c.get("name") == NOMBRE_COLECCION and not c.get("archived")]
assert len(colecciones) == 1
id_coleccion = colecciones[0]["id"]
preguntas = elementos_coleccion(mb, id_coleccion, "card")
resultados = []
for numero, (archivo, (titulo, (display_esperado, ajustes))) in enumerate(INDICADORES.items(), 1):
    coincidencias = [p for p in preguntas if p["name"] == titulo]
    assert len(coincidencias) == 1, f"Pregunta ausente o duplicada: I{numero}"
    id_pregunta = coincidencias[0]["id"]
    assert id_pregunta == 39 + numero, f"Cambio de ID en I{numero}"
    pregunta = mb.pedir("GET", f"/api/card/{id_pregunta}")
    consulta = pregunta["dataset_query"]
    sql_guardado = consulta["stages"][0]["native"] if "stages" in consulta else consulta["native"]["query"]
    assert sql_guardado == (RAIZ / "sql/07_indicadores" / archivo).read_text(encoding="utf-8"), "SQL diferente del archivo original"
    assert pregunta["display"] == display_esperado
    for clave, valor in ajustes.items():
        assert pregunta["visualization_settings"].get(clave) == valor, f"Configuracion inesperada: I{numero} {clave}"
    respuesta = mb.pedir("POST", f"/api/card/{id_pregunta}/query", {})
    assert respuesta["status"] == "completed", f"Consulta fallida: I{numero}"
    datos = respuesta["data"]
    df = pd.DataFrame(datos["rows"], columns=[c["name"] for c in datos["cols"]])
    anios = set()
    for columna in ("anio", "serie", "mes"):
        if columna in df:
            for valor in df[columna].dropna().astype(str):
                anios.update(int(a) for a in re.findall(r"\b20\d{2}\b", valor))
    assert anios == {2024, 2025, 2026}, f"Anios inesperados: I{numero}"
    series = sorted(df["serie"].unique().tolist()) if "serie" in df else []
    if numero == 2:
        assert series == ["green 2025 vs 2024", "green 2026 vs 2025", "yellow 2025 vs 2024", "yellow 2026 vs 2025"]
    resultados.append({"indicador": f"I{numero}", "id": id_pregunta, "estado": "Actualizada y verificada",
                       "filas": len(df), "anios": ", ".join(map(str, sorted(anios))),
                       "dimensiones": ", ".join(ajustes["graph.dimensions"]), "series": "; ".join(series)})
    print(resultados[-1])

tableros = [t for t in elementos_coleccion(mb, id_coleccion, "dashboard") if t["name"] == NOMBRE_TABLERO]
assert len(tableros) == 1 and tableros[0]["id"] == 2
tablero = mb.pedir("GET", "/api/dashboard/2")
ids_incluidos = [t["card_id"] for t in tablero["dashcards"] if t.get("card_id")]
assert len(ids_incluidos) == 11 and set(ids_incluidos) == set(range(40, 51))
print("Dashboard 2: once preguntas originales, una vez cada una; cinco secciones.")
display(pd.DataFrame(resultados))

{'indicador': 'I1', 'id': 40, 'estado': 'Actualizada y verificada', 'filas': 32, 'anios': '2024, 2025, 2026', 'dimensiones': 'mes, anio', 'series': ''}


{'indicador': 'I2', 'id': 41, 'estado': 'Actualizada y verificada', 'filas': 40, 'anios': '2024, 2025, 2026', 'dimensiones': 'mes, serie', 'series': 'green 2025 vs 2024; green 2026 vs 2025; yellow 2025 vs 2024; yellow 2026 vs 2025'}


{'indicador': 'I3', 'id': 42, 'estado': 'Actualizada y verificada', 'filas': 32, 'anios': '2024, 2025, 2026', 'dimensiones': 'mes, anio', 'series': ''}


{'indicador': 'I4', 'id': 43, 'estado': 'Actualizada y verificada', 'filas': 18, 'anios': '2024, 2025, 2026', 'dimensiones': 'anio, componente', 'series': ''}


{'indicador': 'I5', 'id': 44, 'estado': 'Actualizada y verificada', 'filas': 128, 'anios': '2024, 2025, 2026', 'dimensiones': 'mes, metodo_pago', 'series': ''}


...

{'indicador': 'I9', 'id': 48, 'estado': 'Actualizada y verificada', 'filas': 30, 'anios': '2024, 2025, 2026', 'dimensiones': 'zona, anio', 'series': ''}


{'indicador': 'I10', 'id': 49, 'estado': 'Actualizada y verificada', 'filas': 18, 'anios': '2024, 2025, 2026', 'dimensiones': 'aeropuerto, serie', 'series': '2024 % ingresos; 2024 % viajes; 2025 % ingresos; 2025 % viajes; 2026 % ingresos; 2026 % viajes'}


{'indicador': 'I11', 'id': 50, 'estado': 'Actualizada y verificada', 'filas': 64, 'anios': '2024, 2025, 2026', 'dimensiones': 'mes, tipo', 'series': ''}


Dashboard 2: once preguntas originales, una vez cada una; cinco secciones.


,indicador,id,estado,filas,anios,dimensiones,series
0,I1,40,Actualizada y verificada,32,"2024, 2025, 2026","mes, anio",
1,I2,41,Actualizada y verificada,40,"2024, 2025, 2026","mes, serie",green 2025 vs 2024; green 2026 vs 2025; yellow...
2,I3,42,Actualizada y verificada,32,"2024, 2025, 2026","mes, anio",
3,I4,43,Actualizada y verificada,18,"2024, 2025, 2026","anio, componente",
4,I5,44,Actualizada y verificada,128,"2024, 2025, 2026","mes, metodo_pago",
5,I6,45,Actualizada y verificada,32,"2024, 2025, 2026","mes, anio",
6,I7,46,Actualizada y verificada,144,"2024, 2025, 2026","hora, serie",2024 fin de semana; 2024 laborable; 2025 fin d...
7,I8,47,Actualizada y verificada,72,"2024, 2025, 2026","hora, anio",
8,I9,48,Actualizada y verificada,30,"2024, 2025, 2026","zona, anio",
9,I10,49,Actualizada y verificada,18,"2024, 2025, 2026","aeropuerto, serie",2024 % ingresos; 2024 % viajes; 2025 % ingreso...





| Indicador | Filas | Representacion de los años |
|-----------|------:|----------------------------|
| I1 | 32 | Series 2024, 2025 y 2026. |
| I2 | 40 | Series yellow/green 2025 vs 2024 y yellow/green 2026 vs 2025. |
| I3 | 32 | Series 2024, 2025 y 2026. |
| I4 | 18 | Anios 2024, 2025 y 2026 en el eje horizontal; componentes como series. |
| I5 | 128 | Fechas mensuales de 2024, 2025 y 2026 en el eje horizontal; metodos de pago como series. |
| I6 | 32 | Series 2024, 2025 y 2026. |
| I7 | 144 | Seis series: cada anio separado entre laborable y fin de semana. |
| I8 | 72 | Series 2024, 2025 y 2026. |
| I9 | 30 | Series 2024, 2025 y 2026 para las diez zonas seleccionadas. |
| I10 | 18 | Seis series: cada anio separado entre % viajes y % ingresos. |
| I11 | 64 | Fechas mensuales de 2024, 2025 y 2026 en el eje horizontal; tipos de taxi como series. |


## 8.7 Consultas utilizadas para obtener los resultados

### I1 - Viajes por mes

**Archivo SQL:** `sql/07_indicadores/01_viajes_por_mes.sql`.

**Objetivo:** Contar los viajes mensuales por anio, usando todos los registros yellow y green.

**Aporte al Ejercicio 8:** Aporta el volumen por mes y los conteos usados para resumir enero-agosto y comparar demanda.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 1 - Viajes por mes
-- Pregunta: P1. ¿Cuantos viajes se realizan cada mes y como se compara la
--           demanda de un anio con la del otro?
-- Indicador: cantidad de viajes registrados por mes calendario, una serie
--           por anio (yellow + green). Se usa la tabla viajes completa (no
--           viajes_validos): un registro con una hora o distancia mal
--           capturada sigue siendo un viaje realizado.
-- Fuente:   data/processed/taxis.duckdb, tabla viajes (scripts/materializar.py)
-- Visualizacion: lineas; x = mes, y = viajes, serie = anio.
SELECT
    CAST(substr(mes_archivo, 6, 2) AS INTEGER)   AS mes,
    CAST(anio AS VARCHAR)                        AS anio,
    count(*)                                     AS viajes
FROM viajes
GROUP BY ALL
ORDER BY anio, mes;

```


### I2 - Variacion interanual

**Archivo SQL:** `sql/07_indicadores/02_variacion_interanual.sql`.

**Objetivo:** Comparar viajes del mismo mes y tipo de taxi con el anio anterior disponible mediante lag().

**Aporte al Ejercicio 8:** Aporta viajes, viajes_base y variacion_pct; genera 2025 vs 2024 y 2026 vs 2025.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 2 - Variacion interanual de viajes por tipo de taxi
-- Pregunta: P2. ¿La demanda crece o cae respecto al mismo mes del anio
--           anterior disponible, y evolucionan igual yellow y green?
-- Indicador: (viajes del mes / viajes del mismo mes del anio anterior
--           disponible - 1) x 100, por tipo de taxi. Comparar el mismo mes
--           elimina la estacionalidad.
-- Fuente:   data/processed/taxis.duckdb, tabla viajes
-- Nota:     el "anio anterior disponible" es el anio cargado inmediatamente
--           anterior (hoy 2026 frente a 2024; al agregar 2025 en el
--           Ejercicio 8 seran 2025 vs 2024 y 2026 vs 2025, sin cambiar la
--           consulta).
-- Visualizacion: barras; x = mes, y = variacion_pct, serie = serie (tipo + comparacion).
WITH por_mes AS (
    SELECT
        tipo,
        anio,
        CAST(substr(mes_archivo, 6, 2) AS INTEGER)   AS mes,
        count(*)                                     AS viajes
    FROM viajes
    GROUP BY ALL
),
comparados AS (
    SELECT
        *,
        lag(anio)   OVER (PARTITION BY tipo, mes ORDER BY anio)   AS anio_base,
        lag(viajes) OVER (PARTITION BY tipo, mes ORDER BY anio)   AS viajes_base
    FROM por_mes
)
SELECT
    mes,
    tipo || ' ' || anio || ' vs ' || anio_base           AS serie,
    viajes_base,
    viajes,
    round(100.0 * viajes / viajes_base - 100, 1)         AS variacion_pct
FROM comparados
WHERE viajes_base IS NOT NULL
ORDER BY serie DESC, mes;

```


### I3 - Costo tipico del viaje

**Archivo SQL:** `sql/07_indicadores/03_costo_por_viaje.sql`.

**Objetivo:** Obtener medianas mensuales de total, tarifa y distancia de viajes validos yellow con total positivo.

**Aporte al Ejercicio 8:** Aporta las medianas mensuales de costo comparadas entre 2024, 2025 y 2026.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 3 - Costo tipico de un viaje por mes
-- Pregunta: P3. ¿Cuanto paga un pasajero por un viaje tipico y como cambia
--           ese monto en el tiempo?
-- Indicador: mediana de total_amount por viaje, por mes y anio (taxis
--           amarillos). Se usa la mediana porque la distribucion de montos es
--           asimetrica (viajes de aeropuerto y largos elevan el promedio).
-- Fuente:   data/processed/taxis.duckdb, vista viajes_validos
-- Nota:     solo yellow (95%+ de los viajes): mezclar con green, que tiene
--           otra tarifa y otra cobertura, cambiaria la mediana por cambios de
--           composicion y no de precio. Se excluyen montos en cero.
-- Visualizacion: lineas; x = mes, y = total_mediano_usd, serie = anio.
SELECT
    CAST(substr(mes_archivo, 6, 2) AS INTEGER)   AS mes,
    CAST(anio AS VARCHAR)                        AS anio,
    round(median(total), 2)                      AS total_mediano_usd,
    round(median(tarifa), 2)                     AS tarifa_mediana_usd,
    round(median(distancia), 2)                  AS distancia_mediana_mi
FROM viajes_validos
WHERE tipo = 'yellow'
  AND total > 0
GROUP BY ALL
ORDER BY anio, mes;

```


### I4 - Composicion del cobro

**Archivo SQL:** `sql/07_indicadores/04_composicion_del_cobro.sql`.

**Objetivo:** Descomponer el cobro promedio de yellow en seis componentes, sobre meses comunes.

**Aporte al Ejercicio 8:** Aporta 18 filas (seis componentes por tres anios) para verificar y actualizar I4 en 8.3 y 8.4; no se uso en el analisis seleccionado de 8.5.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 4 - Composicion del cobro promedio por viaje
-- Pregunta: P4. ¿Que componentes explican lo que se cobra por un viaje y
--           cuanto aporta el cargo por congestion (CBD) introducido en 2025?
-- Indicador: promedio por viaje, en USD, de cada componente de total_amount:
--           tarifa base, propina, peajes, cargo de aeropuerto, cargo CBD y
--           otros recargos (extra, MTA, mejora, congestion_surcharge = el
--           resto del total).
-- Fuente:   data/processed/taxis.duckdb, vista viajes_validos
-- Nota:     solo yellow y solo los meses publicados en TODOS los anios
--           cargados (hoy enero-agosto), para comparar periodos equivalentes.
--           Los cargos ausentes (NULL, p. ej. CBD en 2024) cuentan como 0.
-- Visualizacion: barras apiladas; x = anio, y = usd_promedio, serie = componente.
WITH meses_comunes AS (
    SELECT substr(mes_archivo, 6, 2) AS mes
    FROM viajes
    GROUP BY mes
    HAVING count(DISTINCT anio) = (SELECT count(DISTINCT anio) FROM viajes)
),
promedios AS (
    SELECT
        CAST(anio AS VARCHAR)                                              AS anio,
        avg(tarifa)                                                        AS "1 Tarifa base",
        avg(propina)                                                       AS "2 Propina",
        avg(peajes)                                                        AS "3 Peajes",
        avg(coalesce(cargo_aeropuerto, 0))                                 AS "4 Cargo aeropuerto",
        avg(coalesce(cargo_cbd, 0))                                        AS "5 Cargo congestion CBD",
        avg(total - tarifa - propina - peajes
            - coalesce(cargo_aeropuerto, 0) - coalesce(cargo_cbd, 0))      AS "6 Otros recargos"
    FROM viajes_validos
    WHERE tipo = 'yellow'
      AND substr(mes_archivo, 6, 2) IN (SELECT mes FROM meses_comunes)
    GROUP BY anio
)
SELECT anio, componente, round(usd_promedio, 2) AS usd_promedio
FROM (UNPIVOT promedios ON COLUMNS(* EXCLUDE (anio)) INTO NAME componente VALUE usd_promedio)
ORDER BY anio, componente;

```


### I5 - Metodos de pago

**Archivo SQL:** `sql/07_indicadores/05_metodos_de_pago.sql`.

**Objetivo:** Contar viajes por mes y categoria de pago y calcular su participacion mensual.

**Aporte al Ejercicio 8:** Aporta conteos por metodo; estos se sumaron posteriormente para calcular las participaciones de enero-agosto.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 5 - Participacion de cada metodo de pago por mes
-- Pregunta: P5. ¿Como pagan los pasajeros y esta cambiando la mezcla de
--           metodos de pago?
-- Indicador: % de los viajes de cada mes pagados con cada metodo (yellow +
--           green). Tarjeta, efectivo y Flex Fare se muestran por separado; el
--           resto (sin cargo, disputa, desconocido, anulado) se agrupa.
-- Fuente:   data/processed/taxis.duckdb, tabla viajes + tabla metodos_pago
-- Nota:     se usa viajes (no viajes_validos), igual que en el Ejercicio 4:
--           los filtros de validez eliminan casi todas las disputas. En green
--           el equivalente a Flex Fare llega con payment_type NULL.
-- Visualizacion: barras apiladas al 100%; x = mes (YYYY-MM), serie = metodo.
SELECT
    v.mes_archivo                                                      AS mes,
    CASE
        WHEN m.metodo_pago IN ('Tarjeta', 'Efectivo', 'Flex Fare') THEN m.metodo_pago
        WHEN v.tipo_pago IS NULL THEN 'Flex Fare'
        ELSE 'Otros'
    END                                                                AS metodo_pago,
    count(*)                                                           AS viajes,
    round(100.0 * count(*) / sum(count(*)) OVER (PARTITION BY v.mes_archivo), 2) AS pct_del_mes
FROM viajes AS v
LEFT JOIN metodos_pago AS m USING (tipo_pago)
GROUP BY v.mes_archivo, 2
ORDER BY mes, metodo_pago;

```


### I6 - Viajes con propina

**Archivo SQL:** `sql/07_indicadores/06_propina_con_tarjeta.sql`.

**Objetivo:** Calcular la proporcion mensual de viajes validos pagados con tarjeta que incluyen propina.

**Aporte al Ejercicio 8:** Aporta pct_con_propina y viajes_tarjeta; 8.5 utilizo el indicador principal y promedios simples de sus porcentajes mensuales. Nota autorizada: el encabezado describe pct_propina_promedio como limitado a [0,100]%, pero least(...,1) solo limita explicitamente el extremo superior; no aplica un minimo de 0%.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 6 - Propina en pagos con tarjeta
-- Pregunta: P6. ¿Que tan generosas son las propinas y cambian entre anios?
-- Indicador: % de los viajes pagados con tarjeta que incluyen propina, por
--           mes y anio. Como referencia se agrega la propina promedio como %
--           de lo cobrado antes de ella (recortada a [0, 100]%).
-- Decision: la propina MEDIANA no sirve como indicador: es exactamente 20%
--           en todos los meses de ambos anios (la opcion sugerida por la
--           pantalla de pago), mientras que la proporcion de pasajeros que
--           deja propina si cambia.
-- Fuente:   data/processed/taxis.duckdb, vista viajes_validos
-- Nota:     solo tarjeta (payment_type = 1): en efectivo la propina casi
--           nunca se registra (Ejercicio 4, consulta 14). Yellow + green.
-- Visualizacion: lineas; x = mes, y = pct_con_propina, serie = anio.
SELECT
    CAST(substr(mes_archivo, 6, 2) AS INTEGER)                            AS mes,
    CAST(anio AS VARCHAR)                                                 AS anio,
    count(*)                                                              AS viajes_tarjeta,
    round(100.0 * count(*) FILTER (WHERE propina > 0) / count(*), 1)      AS pct_con_propina,
    round(100.0 * avg(least(propina / nullif(total - propina, 0), 1)), 1) AS pct_propina_promedio
FROM viajes_validos
WHERE tipo_pago = 1
GROUP BY ALL
ORDER BY anio, mes;

```


### I7 - Demanda por hora

**Archivo SQL:** `sql/07_indicadores/07_demanda_por_hora.sql`.

**Objetivo:** Calcular viajes promedio por fecha, hora, anio y tipo de dia, para meses comunes.

**Aporte al Ejercicio 8:** Aporta curvas y picos horarios; permite comparar laborables y fines de semana en enero-agosto.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 7 - Demanda promedio por hora del dia
-- Pregunta: P7. ¿En que horas se concentra la demanda en dias laborables y
--           en fines de semana, y se mantiene el patron entre anios?
-- Indicador: viajes promedio que inician en cada hora (viajes de la hora /
--           fechas de ese tipo de dia), por tipo de dia y anio.
-- Fuente:   data/processed/taxis.duckdb, tabla viajes
-- Nota:     solo los meses publicados en todos los anios (hoy enero-agosto)
--           para no mezclar estacionalidad, y solo viajes cuya fecha cae en
--           el anio de su archivo. Se divide entre el numero de fechas de
--           cada tipo de dia.
-- Visualizacion: lineas; x = hora, y = viajes_promedio, serie = serie (anio + tipo de dia).
WITH meses_comunes AS (
    -- meses publicados en todos los anios cargados (hoy enero-agosto)
    SELECT substr(mes_archivo, 6, 2) AS mes
    FROM viajes
    GROUP BY mes
    HAVING count(DISTINCT anio) = (SELECT count(DISTINCT anio) FROM viajes)
),
clasificados AS (
    SELECT
        anio,
        CAST(inicio AS DATE)                                                      AS fecha,
        hour(inicio)                                                              AS hora,
        CASE WHEN isodow(inicio) >= 6 THEN 'fin de semana' ELSE 'laborable' END   AS tipo_dia
    FROM viajes
    WHERE year(inicio) = anio
      AND substr(mes_archivo, 6, 2) IN (SELECT mes FROM meses_comunes)
),
fechas AS (
    SELECT anio, tipo_dia, count(DISTINCT fecha) AS num_fechas
    FROM clasificados
    GROUP BY ALL
)
SELECT
    c.hora,
    c.anio || ' ' || c.tipo_dia                       AS serie,
    round(count(*) / any_value(f.num_fechas))         AS viajes_promedio
FROM clasificados AS c
JOIN fechas AS f USING (anio, tipo_dia)
GROUP BY c.hora, c.anio, c.tipo_dia
ORDER BY serie, c.hora;

```


### I8 - Velocidad mediana en Manhattan

**Archivo SQL:** `sql/07_indicadores/08_velocidad_manhattan.sql`.

**Objetivo:** Calcular velocidad mediana por hora y anio de viajes yellow validos dentro de Manhattan, en laborables y meses comunes.

**Aporte al Ejercicio 8:** Aporta medianas horarias en mph, utilizadas para comparar horas concretas y resumir el perfil horario.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 8 - Velocidad mediana de los viajes dentro de Manhattan por hora
-- Pregunta: P8. ¿Que tan congestionado esta el trafico a cada hora y cambio
--           la velocidad despues de la tarifa de congestion de 2025?
-- Indicador: mediana de distancia / duracion (millas por hora) de los
--           viajes yellow que empiezan y terminan en Manhattan, en dias
--           laborables, por hora y anio. La velocidad del taxi es una medida
--           indirecta del trafico.
-- Fuente:   data/processed/taxis.duckdb, vista viajes_validos + tabla zonas
-- Nota:     el cargo CBD (enero de 2025) se cobra al entrar a Manhattan al sur
--           de la calle 60, por lo que Manhattan es donde deberia notarse un
--           cambio. Se excluyen viajes de menos de 1 minuto o con velocidad
--           mayor a 60 mph (errores de registro). Solo meses publicados en
--           todos los anios (hoy enero-agosto): el trafico es estacional.
-- Visualizacion: lineas; x = hora, y = velocidad_mediana_mph, serie = anio.
WITH meses_comunes AS (
    -- meses publicados en todos los anios cargados (hoy enero-agosto)
    SELECT substr(mes_archivo, 6, 2) AS mes
    FROM viajes
    GROUP BY mes
    HAVING count(DISTINCT anio) = (SELECT count(DISTINCT anio) FROM viajes)
)
SELECT
    hour(v.inicio)                                         AS hora,
    CAST(v.anio AS VARCHAR)                                AS anio,
    count(*)                                               AS viajes,
    round(median(v.distancia / (v.duracion_min / 60)), 1)  AS velocidad_mediana_mph
FROM viajes_validos AS v
JOIN zonas AS zo ON zo.id_zona = v.origen
JOIN zonas AS zd ON zd.id_zona = v.destino
WHERE v.tipo = 'yellow'
  AND zo.distrito = 'Manhattan'
  AND zd.distrito = 'Manhattan'
  AND isodow(v.inicio) <= 5
  AND v.duracion_min >= 1
  AND v.distancia / (v.duracion_min / 60) <= 60
  AND substr(v.mes_archivo, 6, 2) IN (SELECT mes FROM meses_comunes)
GROUP BY ALL
ORDER BY anio, hora;

```


### I9 - Zonas de origen

**Archivo SQL:** `sql/07_indicadores/09_zonas_de_origen.sql`.

**Objetivo:** Seleccionar las diez zonas principales del anio mas reciente y comparar su participacion entre anios en meses comunes.

**Aporte al Ejercicio 8:** Aporta conteos y porcentajes de las mismas diez zonas; se sumaron sus porcentajes para describir su concentracion conjunta.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 9 - Zonas con mas viajes de origen
-- Pregunta: P9. ¿Donde se origina la demanda y cambiaron las zonas
--           principales entre anios?
-- Indicador: % de los viajes de cada anio que inicia en cada una de las 10
--           zonas con mas viajes del anio mas reciente.
-- Fuente:   data/processed/taxis.duckdb, tabla viajes + tabla zonas
-- Nota:     solo meses publicados en todos los anios (hoy enero-agosto), y
--           porcentajes sobre el total del anio. Yellow + green.
-- Visualizacion: barras horizontales; y = zona, x = pct_viajes, serie = anio.
WITH meses_comunes AS (
    -- meses publicados en todos los anios cargados (hoy enero-agosto)
    SELECT substr(mes_archivo, 6, 2) AS mes
    FROM viajes
    GROUP BY mes
    HAVING count(DISTINCT anio) = (SELECT count(DISTINCT anio) FROM viajes)
),
por_zona AS (
    SELECT anio, origen AS id_zona, count(*) AS viajes
    FROM viajes
    WHERE substr(mes_archivo, 6, 2) IN (SELECT mes FROM meses_comunes)
    GROUP BY ALL
),
porcentajes AS (
    SELECT *, 100.0 * viajes / sum(viajes) OVER (PARTITION BY anio) AS pct
    FROM por_zona
),
top_reciente AS (
    SELECT id_zona, row_number() OVER (ORDER BY pct DESC) AS posicion
    FROM porcentajes
    WHERE anio = (SELECT max(anio) FROM viajes)
    QUALIFY posicion <= 10
)
SELECT
    t.posicion,
    z.zona || ' (' || z.distrito || ')'      AS zona,
    CAST(p.anio AS VARCHAR)                  AS anio,
    p.viajes,
    round(p.pct, 2)                          AS pct_viajes
FROM top_reciente AS t
JOIN porcentajes AS p USING (id_zona)
JOIN zonas AS z USING (id_zona)
ORDER BY t.posicion, p.anio;

```


### I10 - Aeropuertos

**Archivo SQL:** `sql/07_indicadores/10_aeropuertos.sql`.

**Objetivo:** Calcular participacion en viajes e ingresos de JFK, LaGuardia y Newark por anio, en meses comunes.

**Aporte al Ejercicio 8:** Aporta porcentajes por aeropuerto y medida; se sumaron para obtener el peso conjunto de los tres aeropuertos.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 10 - Peso de los aeropuertos en viajes e ingresos
-- Pregunta: P10. ¿Que parte de los viajes y de los ingresos corresponde a
--           viajes desde o hacia los aeropuertos?
-- Indicador: % de viajes y % de ingresos (total_amount) de los viajes cuyo
--           origen o destino es JFK (132), LaGuardia (138) o Newark (1), por
--           anio y aeropuerto.
-- Fuente:   data/processed/taxis.duckdb, vista viajes_validos + tabla zonas
-- Nota:     solo meses publicados en todos los anios (hoy enero-agosto). Un
--           viaje entre dos aeropuertos se asigna al de origen. Yellow + green.
-- Formato largo (una fila por aeropuerto, anio y medida) para graficar las
-- dos medidas de ambos anios juntas: si % ingresos > % viajes, el viaje de
-- aeropuerto cuesta mas que el promedio.
-- Visualizacion: barras; x = aeropuerto, y = pct, serie = serie (anio + medida).
WITH meses_comunes AS (
    SELECT substr(mes_archivo, 6, 2) AS mes
    FROM viajes
    GROUP BY mes
    HAVING count(DISTINCT anio) = (SELECT count(DISTINCT anio) FROM viajes)
),
clasificados AS (
    SELECT
        anio,
        total,
        CASE
            WHEN origen  IN (1, 132, 138) THEN origen
            WHEN destino IN (1, 132, 138) THEN destino
        END AS id_aeropuerto
    FROM viajes_validos
    WHERE substr(mes_archivo, 6, 2) IN (SELECT mes FROM meses_comunes)
),
por_anio AS (
    SELECT
        anio,
        id_aeropuerto,
        count(*)                                                       AS viajes,
        100.0 * count(*) / sum(count(*)) OVER (PARTITION BY anio)      AS pct_viajes,
        100.0 * sum(total) / sum(sum(total)) OVER (PARTITION BY anio)  AS pct_ingresos
    FROM clasificados
    GROUP BY anio, id_aeropuerto
)
SELECT
    z.zona                                     AS aeropuerto,
    p.anio || ' ' || m.medida                  AS serie,
    round(CASE m.medida WHEN '% viajes' THEN p.pct_viajes ELSE p.pct_ingresos END, 2) AS pct
FROM por_anio AS p
JOIN zonas AS z ON z.id_zona = p.id_aeropuerto
CROSS JOIN (VALUES ('% viajes'), ('% ingresos')) AS m(medida)
ORDER BY aeropuerto, serie;

```


### I11 - Calidad de datos

**Archivo SQL:** `sql/07_indicadores/11_calidad_de_datos.sql`.

**Objetivo:** Comparar conteos totales y validos por mes y tipo de taxi para obtener registros invalidos y su porcentaje.

**Aporte al Ejercicio 8:** Aporta registros, registros_invalidos y pct_invalidos; permite calcular tasas ponderadas del periodo a partir de los conteos. Nota autorizada: el encabezado pregunta por proveedor, pero la consulta agrupa por mes y tipo de taxi, no por proveedor.

**Procedencia:** misma consulta del Ejercicio 7; no es SQL nuevo.

```sql
-- Indicador 11 - Registros que no pasan las reglas de calidad
-- Pregunta: P11. ¿Que tan confiables son los registros de cada mes y
--           proveedor? ¿Empeora o mejora la calidad de los datos?
-- Indicador: % de los registros de cada mes que quedan fuera de
--           viajes_validos (reglas del Ejercicio 3), por tipo de taxi.
-- Fuente:   data/processed/taxis.duckdb, tabla viajes + vista viajes_validos
-- Nota:     un % alto no significa viajes falsos sino registros con datos
--           imposibles (p. ej. hora de fin igual a la de inicio); el
--           Ejercicio 5 mostro que en 2026 el proveedor Helix aporta casi
--           todos los de duracion cero.
-- Visualizacion: lineas; x = mes (YYYY-MM), y = pct_invalidos, serie = tipo.
WITH totales AS (
    SELECT tipo, mes_archivo, count(*) AS registros
    FROM viajes
    GROUP BY ALL
),
validos AS (
    SELECT tipo, mes_archivo, count(*) AS registros_validos
    FROM viajes_validos
    GROUP BY ALL
)
SELECT
    t.mes_archivo                                                            AS mes,
    t.tipo,
    t.registros,
    t.registros - v.registros_validos                                        AS registros_invalidos,
    round(100.0 * (t.registros - v.registros_validos) / t.registros, 2)      AS pct_invalidos
FROM totales AS t
JOIN validos AS v USING (tipo, mes_archivo)
ORDER BY t.tipo DESC, mes;

```


### A1 - Estado de la carga

**Indicadores relacionados:** Todos I1-I11.

**Archivo o lugar de ejecucion:** notebooks/08_incorporacion2025.ipynb y scripts/materializar.py; tambien ejecutada directamente en la inspeccion inicial.

**Objetivo y aporte:** Verificar anios, archivos, registros y periodo de la carga. Se utilizo antes y despues de ampliar la base.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT * FROM carga
```


### A2 - Conteo inicial por anio

**Indicadores relacionados:** I1 y cobertura de I1-I11.

**Archivo o lugar de ejecucion:** Ejecucion directa en Docker durante 8.3; sin archivo SQL independiente.

**Objetivo y aporte:** Detectar que la base anterior solo tenia 2024 y 2026 y establecer sus conteos iniciales.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT anio, count(*) AS filas FROM viajes GROUP BY anio ORDER BY anio
```


### A3 - Anios de la base nueva

**Indicadores relacionados:** Todos I1-I11.

**Archivo o lugar de ejecucion:** Ejecucion directa en Docker durante 8.3; sin archivo SQL independiente.

**Objetivo y aporte:** Confirmar que carga registra [2024,2025,2026] antes de activar la base ampliada.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT anios FROM carga
```


### A4 - Conservacion de registros anteriores

**Indicadores relacionados:** I1 y cobertura de I1-I11.

**Archivo o lugar de ejecucion:** Ejecucion directa en Docker durante 8.3; sin archivo SQL independiente.

**Objetivo y aporte:** Comprobar que se conservan 71,870,407 registros de 2024 y 2026.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT count(*) FROM viajes WHERE anio IN (2024,2026)
```


### A5 - Conteo de registros incorporados

**Indicadores relacionados:** I1 y cobertura de I1-I11.

**Archivo o lugar de ejecucion:** Ejecucion directa en Docker durante 8.3; sin archivo SQL independiente.

**Objetivo y aporte:** Comprobar la incorporacion de 49,313,977 registros de 2025.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT count(*) FROM viajes WHERE anio=2025
```


### A6 - Cobertura y registros por anio

**Indicadores relacionados:** Todos I1-I11.

**Archivo o lugar de ejecucion:** notebooks/08_incorporacion2025.ipynb, celda ej8-3-consultas.

**Objetivo y aporte:** Confirmar registros y meses disponibles: doce meses en 2024 y 2025 y ocho en 2026.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT anio, count(*) AS registros, count(DISTINCT mes_archivo) AS meses FROM viajes GROUP BY anio ORDER BY anio
```


### A7 - Meses comunes

**Indicadores relacionados:** I4, I7, I8, I9 e I10; comparabilidad general.

**Archivo o lugar de ejecucion:** notebooks/08_incorporacion2025.ipynb, celda ej8-3-consultas.

**Objetivo y aporte:** Comprobar que enero-agosto son los meses presentes en los tres anios.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT substr(mes_archivo, 6, 2) AS mes FROM viajes GROUP BY mes HAVING count(DISTINCT anio) = (SELECT count(DISTINCT anio) FROM viajes) ORDER BY mes
```


### A8 - Cobertura consultada para 8.5

**Indicadores relacionados:** Indicadores seleccionados en 8.5.

**Archivo o lugar de ejecucion:** Ejecucion directa en Docker durante 8.5; sin archivo SQL independiente.

**Objetivo y aporte:** Confirmar nuevamente doce meses para 2024/2025 y ocho para 2026 antes de analizar sus resultados.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT anio,count(DISTINCT mes_archivo) AS meses FROM viajes GROUP BY anio ORDER BY anio
```


### A9 - Conteo de filas y archivos de materializacion

**Indicadores relacionados:** Todos I1-I11.

**Archivo o lugar de ejecucion:** scripts/materializar.py; SQL embebido existente.

**Objetivo y aporte:** Obtener 121,184,384 registros y 64 combinaciones tipo-mes_archivo; el conteo se registra en carga.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql
SELECT count(*), count(DISTINCT tipo || mes_archivo) FROM viajes
```


### A10 - Resumen por tipo y anio del materializador

**Indicadores relacionados:** I1, I2 y cobertura de I1-I11.

**Archivo o lugar de ejecucion:** scripts/materializar.py; SQL embebido existente.

**Objetivo y aporte:** Mostrar los registros cargados por tipo de taxi y anio al regenerar la base.

**Procedencia:** consulta auxiliar, no una consulta de indicador del Ejercicio 7. Se documenta la sentencia realmente ejecutada, no una consulta creada para esta seccion.

```sql

        SELECT tipo, anio, count(*) AS viajes
        FROM viajes GROUP BY ALL ORDER BY tipo DESC, anio
    
```


### Calculos posteriores: 8.5 y 8.6, sin SQL nuevo


| Indicador | Tratamiento posterior realmente utilizado |
|-----------|------------------------------------------|
| I1 | Filtrar mes <= 8; sumar viajes por anio y calcular 100 * (total posterior / total anterior - 1). De aqui salen 26,831,594, 31,954,356 y 30,040,469 viajes; +19.09%, -5.99% y +11.96% entre los periodos indicados. |
| I2 | Filtrar mes <= 8; sumar viajes y viajes_base por serie y recalcular la variacion sobre esos totales. No se promedio variacion_pct mensual. De aqui salen yellow +19.59%/-5.87% y green -10.26%/-15.28%. |
| I3 | Filtrar mes <= 8; comparar medianas mensuales y calcular el promedio simple de ocho medianas: USD 20.90, 21.63 y 23.62. No es la mediana conjunta de enero-agosto. Los cambios de 3.52% y 9.16% corresponden a ese promedio. |
| I5 | Extraer el anio del prefijo de mes y filtrar enero-agosto; sumar viajes por anio y metodo; calcular cada suma / total del anio * 100. Las participaciones de Flex Fare (9.43%, 23.07%, 25.85%) son ponderadas por conteos, no promedios simples de pct_del_mes. |
| I6 | Filtrar mes <= 8; comparar pct_con_propina por mes y promediar los ocho porcentajes mensuales: 94.36%, 93.29%, 91.13%. No es una tasa conjunta ponderada por viajes_tarjeta. No se utilizo pct_propina_promedio para los hallazgos. |
| I7 | Sobre resultados ya restringidos a meses comunes, identificar maximos por serie con idxmax(), seleccionar horas 0,8,18,23 y calcular cambios relativos de los valores comparados. Se calcularon tambien promedio y suma de valores horarios para explorar el perfil; no son conteos brutos de viajes del periodo. |
| I8 | Sobre meses comunes, seleccionar horas 0,8,12,17,18,23; identificar minimos/maximos por anio y calcular el promedio simple de 24 medianas horarias: 9.88, 9.90, 9.56 mph. No es la mediana conjunta de velocidad de todos los viajes. |
| I9 | Comparar porcentajes y conteos devueltos para las mismas diez zonas seleccionadas por 2026; sumar pct_viajes de esas zonas: aproximadamente 36.24%, 33.76%, 33.14%. La suma utiliza porcentajes previamente redondeados. No son tres top independientes. |
| I10 | Sumar pct de JFK, LaGuardia y Newark por anio y medida: viajes 10.07%, 8.83%, 8.11%; ingresos 27.76%, 24.48%, 20.90%. Son sumas de porcentajes redondeados y describen participacion, no ingresos absolutos. |
| I11 | Extraer anio y filtrar enero-agosto; sumar registros y registros_invalidos por anio/tipo; calcular 100 * suma invalidos / suma registros. Las tasas yellow 3.31%, 9.05%, 4.92% y green 5.55%, 5.18%, 3.82% se obtienen de conteos, no promediando pct_invalidos mensual. |

